# Importing Important Libraries For Extraction and Transformation of Data

In [1]:
import pandas as pd
import numpy as np
import fastf1
import os

## Caching Data from Fast F1

In [2]:
fastf1.Cache.enable_cache("f1_cache")

### Loading Data of 2023,2024,2025 Austrain Grand Prix

In [3]:
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

sessions = {}

for year in range(2023, 2026):
    try:
        session = fastf1.get_session(year, 'Australia', 'R')
        session.load()
        sessions[year] = session
        print(f"{year} loaded successfully")
    except Exception as e:
        print(f"Failed for {year}: {e}")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

2023 loaded successfully


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

2024 loaded successfully


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

2025 loaded successfully


## Fecthing important Columns

In [4]:
laps={}
for year in range(2023, 2026):
    
    laps[year] = sessions[year].laps[["Driver","LapTime","Sector1Time","Sector2Time","Sector3Time"]].copy()
    

In [5]:
laps[2023].head()

,Driver,LapTime,Sector1Time,Sector2Time,Sector3Time
0,VER,0 days 00:01:41.571000,NaT,0 days 00:00:18.707000,0 days 00:00:46.357000
1,VER,0 days 00:02:12.105000,0 days 00:00:48.846000,0 days 00:00:31.614000,0 days 00:00:51.645000
2,VER,0 days 00:02:10.657000,0 days 00:00:46.836000,0 days 00:00:29.178000,0 days 00:00:54.643000
3,VER,0 days 00:01:23.391000,0 days 00:00:28.900000,0 days 00:00:18.326000,0 days 00:00:36.165000
4,VER,0 days 00:01:23.104000,0 days 00:00:28.935000,0 days 00:00:18.347000,0 days 00:00:35.822000


## Dropping all the Null Values


In [6]:
for year in range (2023,2026):
    laps[year].dropna(inplace=True)

## Converting the Times in Seconds 

In [7]:
for year in range (2023,2026):
    for col in["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
        laps[year][f"{col} (s)"] = laps[year][col].dt.total_seconds()

## Taking Average of all the Times

In [8]:
sector_times = {}

for year in range(2023,2026):
    sector_times[year] = laps[year].groupby("Driver").agg({
        "Sector1Time (s)":"mean",
        "Sector2Time (s)":"mean",
        "Sector3Time (s)":"mean"
    }).reset_index()

## Add New Columns Like Total Sector Time and Year

In [9]:
for year in range(2023, 2026):
    sector_times[year][f"TotalSectorTime_{year} (s)"]=(
        sector_times[year]["Sector1Time (s)"]+
        sector_times[year]["Sector2Time (s)"]+
        sector_times[year]["Sector3Time (s)"]
    )
    sector_times[year]["Year"] = year

In [10]:
sector_times[2024]

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime_2024 (s),Year
0,ALB,29.373893,18.522982,36.676339,84.573214,2024
1,ALO,29.153140,18.555982,36.735298,84.444421,2024
2,BOT,29.563875,18.826607,36.866768,85.257250,2024
3,GAS,29.606286,18.629214,36.862571,85.098071,2024
4,HAM,29.553357,18.145857,36.924929,84.624143,2024
5,HUL,29.372895,18.810175,36.632298,84.815368,2024
6,LEC,28.882860,18.224632,36.027404,83.134895,2024
7,MAG,29.324036,18.430304,36.807393,84.561732,2024
8,NOR,28.914263,18.278035,36.017579,83.209877,2024
9,OCO,29.603321,18.689143,37.145732,85.438196,2024


## Concating all the tables

In [11]:
combined_sector_time = pd.concat(sector_times.values(), ignore_index=True)

In [12]:
combined_sector_time

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime_2023 (s),Year,TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,ALB,36.661600,22.926600,42.705400,102.293600,2023,NaN,NaN
1,ALO,29.568863,19.144961,36.672275,85.386098,2023,NaN,NaN
2,BOT,30.264860,19.164900,37.449260,86.879020,2023,NaN,NaN
3,DEV,30.642140,19.425180,37.693360,87.760680,2023,NaN,NaN
4,GAS,29.669765,18.958725,36.978529,85.607020,2023,NaN,NaN
5,HAM,30.528673,19.423365,37.291846,87.243885,2023,NaN,NaN
6,HUL,29.815098,19.174549,37.051902,86.041549,2023,NaN,NaN
7,MAG,30.205939,19.053918,37.344633,86.604490,2023,NaN,NaN
8,NOR,29.770627,19.360314,36.842412,85.973353,2023,NaN,NaN
9,OCO,30.065784,19.163157,36.823059,86.052000,2023,NaN,NaN


## Clean Air Race Pace (Best Time From Practice Session Before Qualifying Round.)

In [13]:
clean_air_race_pace = {
    #McLaren
    "NOR":00,               #Lando Norris 🇬🇧
    "PIA":00,               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER":00,               #Max Verstappen 🇳🇱
    "HAD":00,               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC":00,               #Charles Leclerc 🇲🇨
    "HAM":00,               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS":00,               #Geaorge Russell 🇬🇧
    "ANT":00,               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO":00,               #Fernando Alonso 🇪🇸
    "STR":00,               #Lance Stroll 🇨🇦
    #Williams
    "ALB":00,               #Alexander Albon 🇹🇭
    "SAI":00,               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO":00,               #Esteban Ocon 🇫🇷
    "BEA":00,               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS":00,               #Piette Gasly 🇫🇷
    "COL":00,               #Franco Colapinto 🇦🇷
    #Audi
    "HUL":00,               #Nico Hulkenberg 🇩🇪
    "BOR":00,               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER":00,               #Sergio Perez 🇲🇽
    "BOT":00,               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW":00,               #Liam Lawson 🇳🇿
    "LIN":00                #Arvid Lindblad 🇬🇧
}

## List of the Current Drivers

In [14]:
Driver = [
    #McLaren
    "NOR",               #Lando Norris 🇬🇧
    "PIA",               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER",               #Max Verstappen 🇳🇱
    "HAD",               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC",               #Charles Leclerc 🇲🇨
    "HAM",               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS",               #Geaorge Russell 🇬🇧
    "ANT",               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO",               #Fernando Alonso 🇪🇸
    "STR",               #Lance Stroll 🇨🇦
    #Williams
    "ALB",               #Alexander Albon 🇹🇭
    "SAI",               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO",               #Esteban Ocon 🇫🇷
    "BEA",               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS",               #Piette Gasly 🇫🇷
    "COL",               #Franco Colapinto 🇦🇷
    #Audi
    "HUL",               #Nico Hulkenberg 🇩🇪
    "BOR",               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER",               #Sergio Perez 🇲🇽
    "BOT",               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW",               #Liam Lawson 🇳🇿
    "LIN"                #Arvid Lindblad 🇬🇧

]

## Qualifying Time 

In [15]:
qualifying_2026 = pd.DataFrame({
    "Driver":Driver,
    "QualifyingTime (s)":[
        #McLaren
        00,                 #NOR
        00,                 #PIA
        #Red Bull Racing
        00,                 #VER
        00,                 #HAD
        #Ferrari
        00,                 #LEC
        00,                 #HAM
        #Mercedes
        00,                 #RUS
        00,                 #ANT
        #Aston Martin Aramco
        00,                 #ALO
        00,                 #STR
        #Williams
        00,                 #ALB
        00,                 #SAI
        #Haas
        00,                 #OCO
        00,                 #BEA
        #Alpine
        00,                 #GAS
        00,                 #COL
        #Audi
        00,                 #HUL
        00,                 #BOR
        #Cadillac
        00,                 #PER
        00,                 #BOT
        #Racing Bulls
        00,                 #LAW
        00                  #LIN
    ]
})

## Combining Qualifying Time and Clean Air Race Pace 

In [16]:
qualifying_2026["CleanAirRacePace (s)"] = qualifying_2026["Driver"].map(clean_air_race_pace)

In [17]:
qualifying_2026

,Driver,QualifyingTime (s),CleanAirRacePace (s)
0,NOR,0,0
1,PIA,0,0
2,VER,0,0
3,HAD,0,0
4,LEC,0,0
5,HAM,0,0
6,RUS,0,0
7,ANT,0,0
8,ALO,0,0
9,STR,0,0


## Team Points(Currently Zero Due to Start of Season)

In [18]:
team_points = {
    "McLaren":00,
    "Mercedes":00,
    "Red Bull":00,
    "Williams":00,
    "Ferrari":00,
    "Haas":00,
    "Aston Martin":00,
    "Racing Bulls":00,
    "Alpine":00,
    "Audi":00,
    "Cardillac":00
}

## Scaling the Team Points (For Future Use)

In [19]:
from sklearn.preprocessing import MinMaxScaler

teams = list(team_points.keys())
points = np.array(list(team_points.values())).reshape(-1,1)

scaler = MinMaxScaler()
scaled_points = scaler.fit_transform(points)

team_performance_score = dict(zip(teams, scaled_points.flatten()))

## Mapping Drivers with their Respective Teams

In [20]:
driver_to_team = {
    "NOR":"McLaren",
    "PIA":"McLaren",

    "VER":"Red Bull",
    "HAD":"Red Bull",

    "LEC":"Ferrari",
    "HAM":"Ferrari",

    "RUS":"Mercedes",
    "ANT":"Mercedes",

    "ALO":"Aston Martin",
    "STR":"Aston Martin",

    "ALB":"Williams",
    "SAI":"Williams",

    "OCO":"Haas",
    "BEA":"Haas",

    "GAS":"Alpine",
    "COL":"Alpine",

    "HUL":"Audi",
    "BOR":"Audi",

    "PER":"Cardillac",
    "BOT":"Cardillac",
    
    "LAW":"Racing Bulls",
    "LIN":"Racing Bulls"  
}

## Combining Team and Team Performance data to Qualifying 2026 table

In [21]:
qualifying_2026["Team"] = qualifying_2026["Driver"].map(driver_to_team)
qualifying_2026["TeamPerformanceScore"] = qualifying_2026["Team"].map(team_performance_score)

In [22]:
qualifying_2026

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore
0,NOR,0,0,McLaren,0.0
1,PIA,0,0,McLaren,0.0
2,VER,0,0,Red Bull,0.0
3,HAD,0,0,Red Bull,0.0
4,LEC,0,0,Ferrari,0.0
5,HAM,0,0,Ferrari,0.0
6,RUS,0,0,Mercedes,0.0
7,ANT,0,0,Mercedes,0.0
8,ALO,0,0,Aston Martin,0.0
9,STR,0,0,Aston Martin,0.0


## Merging Data of qualifyingTime and Total Sector Time of 2023,2024,2025

In [23]:
merged_data = qualifying_2026.copy()

for year in range(2023, 2026):
    merged_data = merged_data.merge(
        sector_times[year][["Driver", f"TotalSectorTime_{year} (s)"]],
        on="Driver",
        how="left"
    )

In [24]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481
3,HAD,0,0,Red Bull,0.0,NaN,NaN,NaN
4,LEC,0,0,Ferrari,0.0,NaN,83.134895,103.461481
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308
7,ANT,0,0,Mercedes,0.0,NaN,NaN,103.866679
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377


## Filling nan values of 2025 with the average values of 2023 and 2024

### This is done for the drivers whose data are present for 2023 and 2024 but where unable to perform in 2025 due to various reasons.

In [25]:
col_23 = "TotalSectorTime_2023 (s)"
col_24 = "TotalSectorTime_2024 (s)"
col_25 = "TotalSectorTime_2025 (s)"

mask = (
    merged_data[col_25].isna() &
    merged_data[col_23].notna() &
    merged_data[col_24].notna()
)

merged_data.loc[mask, col_25] = (
    merged_data.loc[mask, [col_23, col_24]].mean(axis=1)
)

In [26]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481
3,HAD,0,0,Red Bull,0.0,NaN,NaN,NaN
4,LEC,0,0,Ferrari,0.0,NaN,83.134895,103.461481
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308
7,ANT,0,0,Mercedes,0.0,NaN,NaN,103.866679
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377


## Adding a new column IsNewDriver to let the model know we have missing values and compute accordingly

In [27]:
merged_data["IsNewDriver"] = (
    merged_data["TotalSectorTime_2023 (s)"].isna() &
    merged_data["TotalSectorTime_2024 (s)"].isna()
).astype(int)

In [28]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404,0
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353,0
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481,0
3,HAD,0,0,Red Bull,0.0,NaN,NaN,NaN,1
4,LEC,0,0,Ferrari,0.0,NaN,83.134895,103.461481,0
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115,0
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308,0
7,ANT,0,0,Mercedes,0.0,NaN,NaN,103.866679,1
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643,0
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377,0


## Imputing all the NAN values With Global Median

In [29]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

for year in range(2023, 2026):
    col = f"TotalSectorTime_{year} (s)"
    merged_data[[col]] = imputer.fit_transform(
        merged_data[[col]]
    )

In [30]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404,0
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353,0
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481,0
3,HAD,0,0,Red Bull,0.0,86.007451,84.444421,103.042538,1
4,LEC,0,0,Ferrari,0.0,86.007451,83.134895,103.461481,0
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115,0
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308,0
7,ANT,0,0,Mercedes,0.0,86.007451,84.444421,103.866679,1
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643,0
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377,0


In [31]:
merged_data.columns

Index(['Driver', 'QualifyingTime (s)', 'CleanAirRacePace (s)', 'Team',
       'TeamPerformanceScore', 'TotalSectorTime_2023 (s)',
       'TotalSectorTime_2024 (s)', 'TotalSectorTime_2025 (s)', 'IsNewDriver'],
      dtype='object')

# Lets try to predict 2025 using 2023 and 2024 data and make our model accurate

In [32]:
merged_data1 = merged_data.copy()

In [33]:
merged_data1

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404,0
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353,0
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481,0
3,HAD,0,0,Red Bull,0.0,86.007451,84.444421,103.042538,1
4,LEC,0,0,Ferrari,0.0,86.007451,83.134895,103.461481,0
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115,0
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308,0
7,ANT,0,0,Mercedes,0.0,86.007451,84.444421,103.866679,1
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643,0
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377,0


In [35]:
qualifying_data_2025 = fastf1.get_session(2025,'Australia','Q')

In [36]:
qualifying_data_2025.load()

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            IN

In [37]:
qualifying_data_2025.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:16:54.443000,NOR,4,0 days 00:01:50.421000,1.0,1.0,0 days 00:15:08.068000,NaT,0 days 00:00:48.877000,0 days 00:00:24.492000,...,True,McLaren,0 days 00:15:08.068000,2025-03-15 05:02:39.667,1,NaN,False,,False,False
1,0 days 00:18:10.446000,NOR,4,0 days 00:01:16.003000,2.0,1.0,NaT,NaT,0 days 00:00:26.383000,0 days 00:00:17.285000,...,True,McLaren,0 days 00:16:54.443000,2025-03-15 05:04:26.042,1,NaN,False,,False,True
2,0 days 00:20:12.796000,NOR,4,0 days 00:02:02.350000,3.0,1.0,NaT,0 days 00:19:55.246000,0 days 00:00:37.180000,0 days 00:00:23.115000,...,True,McLaren,0 days 00:18:10.446000,2025-03-15 05:05:42.045,1,NaN,False,,False,False
3,0 days 00:22:09.056000,NOR,4,0 days 00:01:56.260000,4.0,2.0,0 days 00:20:16.711000,NaT,0 days 00:00:52.032000,0 days 00:00:23.199000,...,False,McLaren,0 days 00:20:12.796000,2025-03-15 05:07:44.395,1,NaN,False,,False,False
4,0 days 00:23:24.968000,NOR,4,0 days 00:01:15.912000,5.0,2.0,NaT,NaT,0 days 00:00:26.463000,0 days 00:00:17.116000,...,False,McLaren,0 days 00:22:09.056000,2025-03-15 05:09:40.655,1,NaN,False,,False,True


In [38]:
Quali_2025 = qualifying_data_2025.laps.pick_quicklaps()

Quali = Quali_2025.groupby("Driver")["LapTime"].min().dt.total_seconds()

In [44]:
Quali

Driver
ALB    75.737
ALO    76.288
ANT    76.525
BOR    76.516
DOO    76.315
GAS    75.980
HAD    76.175
HAM    75.919
HUL    76.579
LAW    77.094
LEC    75.755
NOR    75.096
OCO    77.147
PIA    75.180
RUS    75.546
SAI    75.931
STR    76.369
TSU    75.670
VER    75.481
Name: LapTime, dtype: float64

In [63]:
quali_df = Quali.reset_index()

In [64]:
quali_df.columns = ["Driver","QulaifyingTime (s)"]

In [65]:
quali_df

,Driver,QulaifyingTime (s)
0,ALB,75.737
1,ALO,76.288
2,ANT,76.525
3,BOR,76.516
4,DOO,76.315
5,GAS,75.980
6,HAD,76.175
7,HAM,75.919
8,HUL,76.579
9,LAW,77.094


In [66]:
qualifying_2025 = pd.DataFrame({
    "Driver":Driver,
    "QualifyingTime (s)":[
        #McLaren
        00,                 #NOR
        00,                 #PIA
        #Red Bull Racing
        00,                 #VER
        00,                 #HAD
        #Ferrari
        00,                 #LEC
        00,                 #HAM
        #Mercedes
        00,                 #RUS
        00,                 #ANT
        #Aston Martin Aramco
        00,                 #ALO
        00,                 #STR
        #Williams
        00,                 #ALB
        00,                 #SAI
        #Haas
        00,                 #OCO
        00,                 #BEA
        #Alpine
        00,                 #GAS
        00,                 #COL
        #Audi
        00,                 #HUL
        00,                 #BOR
        #Cadillac
        00,                 #PER
        00,                 #BOT
        #Racing Bulls
        00,                 #LAW
        00                  #LIN
    ]
})

In [67]:
qualifying_2025

,Driver,QualifyingTime (s)
0,NOR,0
1,PIA,0
2,VER,0
3,HAD,0
4,LEC,0
5,HAM,0
6,RUS,0
7,ANT,0
8,ALO,0
9,STR,0


In [68]:
qualifying_2025 = qualifying_2025.set_index("Driver")
quali_df = quali_df.set_index("Driver")

qualifying_2025 ["QualifyingTime (s)"] = quali_df["QulaifyingTime (s)"]
qualifying_2025.reset_index(inplace=True)
quali_df.reset_index(inplace=True)

In [69]:
qualifying_2025.fillna(0,inplace=True)

In [70]:
qualifying_2025

,Driver,QualifyingTime (s)
0,NOR,75.096
1,PIA,75.180
2,VER,75.481
3,HAD,76.175
4,LEC,75.755
5,HAM,75.919
6,RUS,75.546
7,ANT,76.525
8,ALO,76.288
9,STR,76.369


In [71]:
practice1 = fastf1.get_session(2025, 'Australia', 'FP1')
practice2 = fastf1.get_session(2025, 'Australia', 'FP2')
practice3 = fastf1.get_session(2025, 'Australia', 'FP3')

In [72]:
practice1.load()
practice2.load()
practice3.load()

core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            IN

In [73]:
practice1.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:17:15.002000,VER,1,0 days 00:01:58.029000,1.0,1.0,0 days 00:15:28.184000,NaT,0 days 00:00:50.194000,0 days 00:00:19.878000,...,True,Red Bull Racing,0 days 00:15:28.184000,2025-03-14 01:30:56.193,1,NaN,False,,False,False
1,0 days 00:18:34.773000,VER,1,0 days 00:01:19.771000,2.0,1.0,NaT,NaT,0 days 00:00:27.951000,0 days 00:00:17.793000,...,True,Red Bull Racing,0 days 00:17:15.002000,2025-03-14 01:32:43.011,1,NaN,False,,False,True
2,0 days 00:20:58.688000,VER,1,0 days 00:02:23.915000,3.0,1.0,NaT,NaT,0 days 00:00:49.204000,0 days 00:00:37.267000,...,True,Red Bull Racing,0 days 00:18:34.773000,2025-03-14 01:34:02.782,1,NaN,False,,False,True
3,0 days 00:22:17.837000,VER,1,0 days 00:01:19.149000,4.0,1.0,NaT,NaT,0 days 00:00:27.343000,0 days 00:00:17.455000,...,True,Red Bull Racing,0 days 00:20:58.688000,2025-03-14 01:36:26.697,1,NaN,False,,False,True
4,0 days 00:24:46.613000,VER,1,0 days 00:02:28.776000,5.0,1.0,NaT,NaT,0 days 00:00:51.751000,0 days 00:00:43.081000,...,True,Red Bull Racing,0 days 00:22:17.837000,2025-03-14 01:37:45.846,1,NaN,False,,False,True


In [74]:
practice1.laps.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [75]:
plaps1 = practice1.laps.pick_quicklaps()

pbest_laps1 = plaps1.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps1)

Driver
ALB    77.713
ALO    77.736
ANT    78.390
BEA    79.312
BOR    78.438
DOO    78.232
GAS    78.505
HAD    77.847
HAM    78.071
HUL    78.586
LAW    78.455
LEC    77.461
NOR    77.252
OCO    79.139
PIA    77.670
RUS    77.716
SAI    77.401
STR    78.057
TSU    78.061
VER    77.696
Name: LapTime, dtype: float64


In [76]:
plaps2 = practice2.laps.pick_quicklaps()

pbest_laps2 = plaps2.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps2)

Driver
ALB    77.302
ALO    77.330
ANT    77.634
BOR    77.847
DOO    77.394
GAS    77.493
HAD    77.019
HAM    76.859
HUL    77.161
LAW    77.640
LEC    76.439
NOR    76.580
OCO    78.034
PIA    76.563
RUS    77.282
SAI    77.302
STR    77.279
TSU    76.784
VER    77.063
Name: LapTime, dtype: float64


In [77]:
plaps3 = practice3.laps.pick_quicklaps()

pbest_laps3 = plaps3.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps3)

Driver
ALB    76.258
ALO    77.270
ANT    76.206
BOR    76.707
DOO    76.993
GAS    76.719
HAD    76.732
HAM    76.378
HUL    77.146
LEC    76.188
NOR    76.597
OCO    77.373
PIA    75.921
RUS    75.960
SAI    76.252
STR    76.948
TSU    76.455
VER    76.002
Name: LapTime, dtype: float64


In [78]:
pbest_laps1 = pbest_laps1.rename("FP1")
pbest_laps2 = pbest_laps2.rename("FP2")
pbest_laps3 = pbest_laps3.rename("FP3")

combined = pd.concat(
    [pbest_laps1, pbest_laps2, pbest_laps3],
    axis=1
)


In [79]:
combined

,FP1,FP2,FP3
Driver,,,
ALB,77.713,77.302,76.258
ALO,77.736,77.330,77.270
ANT,78.390,77.634,76.206
BEA,79.312,NaN,NaN
BOR,78.438,77.847,76.707
DOO,78.232,77.394,76.993
GAS,78.505,77.493,76.719
HAD,77.847,77.019,76.732
HAM,78.071,76.859,76.378


In [90]:
combined["Best_Practice_Time"] = combined.min(axis=1)
combined

,FP1,FP2,FP3,Best_Practice_Time
Driver,,,,
ALB,77.713,77.302,76.258,76.258
ALO,77.736,77.330,77.270,77.270
ANT,78.390,77.634,76.206,76.206
BEA,79.312,NaN,NaN,79.312
BOR,78.438,77.847,76.707,76.707
DOO,78.232,77.394,76.993,76.993
GAS,78.505,77.493,76.719,76.719
HAD,77.847,77.019,76.732,76.732
HAM,78.071,76.859,76.378,76.378


In [84]:
clean_air_race_pace_2025 = {
    #McLaren
    "NOR":00,               #Lando Norris 🇬🇧
    "PIA":00,               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER":00,               #Max Verstappen 🇳🇱
    "HAD":00,               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC":00,               #Charles Leclerc 🇲🇨
    "HAM":00,               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS":00,               #Geaorge Russell 🇬🇧
    "ANT":00,               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO":00,               #Fernando Alonso 🇪🇸
    "STR":00,               #Lance Stroll 🇨🇦
    #Williams
    "ALB":00,               #Alexander Albon 🇹🇭
    "SAI":00,               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO":00,               #Esteban Ocon 🇫🇷
    "BEA":00,               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS":00,               #Piette Gasly 🇫🇷
    "COL":00,               #Franco Colapinto 🇦🇷
    #Audi
    "HUL":00,               #Nico Hulkenberg 🇩🇪
    "BOR":00,               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER":00,               #Sergio Perez 🇲🇽
    "BOT":00,               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW":00,               #Liam Lawson 🇳🇿
    "LIN":00                #Arvid Lindblad 🇬🇧
}

In [87]:
clean_air_df = pd.DataFrame(list(clean_air_race_pace_2025.items()),columns=['Driver','Clean_air_pace_2025'])

In [89]:
clean_air_df

,Driver,Clean_air_pace_2025
0,NOR,0
1,PIA,0
2,VER,0
3,HAD,0
4,LEC,0
5,HAM,0
6,RUS,0
7,ANT,0
8,ALO,0
9,STR,0


In [91]:
clean_air_df = clean_air_df.set_index("Driver")
clean_air_df["Clean_air_pace_2025"] = combined["Best_Practice_Time"]
clean_air_df.reset_index(inplace=True)

In [93]:
clean_air_df.fillna(0,inplace=True)

In [101]:
clean_air_df

,Driver,Clean_air_pace_2025
0,NOR,76.580
1,PIA,75.921
2,VER,76.002
3,HAD,76.732
4,LEC,76.188
5,HAM,76.378
6,RUS,75.960
7,ANT,76.206
8,ALO,77.270
9,STR,76.948


In [95]:
team_points_2025 = {
    "McLaren":27,
    "Mercedes":27,
    "Red Bull":18,
    "Williams":10,
    "Ferrari":5,
    "Haas":00,
    "Aston Martin":8,
    "Racing Bulls":00,
    "Alpine":00,
    "Audi":00,
    "Cardillac":00
}

In [96]:
from sklearn.preprocessing import MinMaxScaler

teams_2025 = list(team_points_2025.keys())
points_2025 = np.array(list(team_points_2025.values())).reshape(-1,1)

scaler = MinMaxScaler()
scaled_points_2025 = scaler.fit_transform(points_2025)

team_performance_score_2025 = dict(zip(teams_2025, scaled_points_2025.flatten()))

In [97]:
qualifying_2025["Team"] = qualifying_2025["Driver"].map(driver_to_team)
qualifying_2025["TeamPerformanceScore"] = qualifying_2025["Team"].map(team_performance_score_2025)

In [100]:
qualifying_2025

,Driver,QualifyingTime (s),Team,TeamPerformanceScore
0,NOR,75.096,McLaren,1.000000
1,PIA,75.180,McLaren,1.000000
2,VER,75.481,Red Bull,0.666667
3,HAD,76.175,Red Bull,0.666667
4,LEC,75.755,Ferrari,0.185185
5,HAM,75.919,Ferrari,0.185185
6,RUS,75.546,Mercedes,1.000000
7,ANT,76.525,Mercedes,1.000000
8,ALO,76.288,Aston Martin,0.296296
9,STR,76.369,Aston Martin,0.296296


In [102]:
clean_air_df = clean_air_df.set_index('Driver')
qualifying_2025 = qualifying_2025.set_index('Driver')

qualifying_2025["Clean_air_race_pace"] = clean_air_df["Clean_air_pace_2025"]

clean_air_df.reset_index(inplace=True)
qualifying_2025.reset_index(inplace=True)

In [110]:
qualifying_2025

,Driver,QualifyingTime (s),Team,TeamPerformanceScore,Clean_air_race_pace
0,NOR,75.096,McLaren,1.000000,76.580
1,PIA,75.180,McLaren,1.000000,75.921
2,VER,75.481,Red Bull,0.666667,76.002
3,HAD,76.175,Red Bull,0.666667,76.732
4,LEC,75.755,Ferrari,0.185185,76.188
5,HAM,75.919,Ferrari,0.185185,76.378
6,RUS,75.546,Mercedes,1.000000,75.960
7,ANT,76.525,Mercedes,1.000000,76.206
8,ALO,76.288,Aston Martin,0.296296,77.270
9,STR,76.369,Aston Martin,0.296296,76.948


In [104]:
merged_data1

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,85.973353,83.209877,103.165404,0
1,PIA,0,0,McLaren,0.0,86.659000,83.711877,101.717353,0
2,VER,0,0,Red Bull,0.0,85.153667,83.607000,103.032481,0
3,HAD,0,0,Red Bull,0.0,86.007451,84.444421,103.042538,1
4,LEC,0,0,Ferrari,0.0,86.007451,83.134895,103.461481,0
5,HAM,0,0,Ferrari,0.0,87.243885,84.624143,103.411115,0
6,RUS,0,0,Mercedes,0.0,93.307714,83.505891,103.283308,0
7,ANT,0,0,Mercedes,0.0,86.007451,84.444421,103.866679,1
8,ALO,0,0,Aston Martin,0.0,85.386098,84.444421,97.136643,0
9,STR,0,0,Aston Martin,0.0,85.691255,84.688246,104.074377,0


In [111]:
merged_data1 = merged_data1.set_index("Driver")
qualifying_2025 = qualifying_2025.set_index("Driver")

merged_data1[["QualifyingTime (s)","CleanAirRacePace (s)", "Team", "TeamPerformanceScore"]] = qualifying_2025[["QualifyingTime (s)","Clean_air_race_pace", "Team", "TeamPerformanceScore"]]

merged_data1.reset_index(inplace=True)
qualifying_2025.reset_index(inplace=True)

In [112]:
merged_data1

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,75.096,76.580,McLaren,1.000000,85.973353,83.209877,103.165404,0
1,PIA,75.180,75.921,McLaren,1.000000,86.659000,83.711877,101.717353,0
2,VER,75.481,76.002,Red Bull,0.666667,85.153667,83.607000,103.032481,0
3,HAD,76.175,76.732,Red Bull,0.666667,86.007451,84.444421,103.042538,1
4,LEC,75.755,76.188,Ferrari,0.185185,86.007451,83.134895,103.461481,0
5,HAM,75.919,76.378,Ferrari,0.185185,87.243885,84.624143,103.411115,0
6,RUS,75.546,75.960,Mercedes,1.000000,93.307714,83.505891,103.283308,0
7,ANT,76.525,76.206,Mercedes,1.000000,86.007451,84.444421,103.866679,1
8,ALO,76.288,77.270,Aston Martin,0.296296,85.386098,84.444421,97.136643,0
9,STR,76.369,76.948,Aston Martin,0.296296,85.691255,84.688246,104.074377,0


In [177]:
merged_data1.columns

Index(['Driver', 'QualifyingTime (s)', 'CleanAirRacePace (s)', 'Team',
       'TeamPerformanceScore', 'TotalSectorTime_2023 (s)',
       'TotalSectorTime_2024 (s)', 'TotalSectorTime_2025 (s)', 'IsNewDriver'],
      dtype='object')

## Defining Features (X) and Target (y)

In [179]:
X = merged_data1[[
    'QualifyingTime (s)', 'CleanAirRacePace (s)', 
       'TeamPerformanceScore', 'TotalSectorTime_2023 (s)',
       'TotalSectorTime_2024 (s)', 'IsNewDriver'
]]

In [180]:
X

,QualifyingTime (s),CleanAirRacePace (s),TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),IsNewDriver
0,63.971,64.324,1.000000,72.480386,71.709254,0
1,64.492,64.442,1.000000,73.628261,71.253629,0
2,64.929,64.534,0.433155,72.228114,71.685729,0
3,65.226,65.547,0.433155,72.695271,72.017786,1
4,64.492,0.000,0.489305,72.277514,72.110986,0
5,64.582,64.790,0.489305,72.695271,71.587143,0
6,64.763,65.018,0.532086,72.680000,71.262814,0
7,65.276,65.053,0.532086,72.695271,72.017786,1
8,65.128,65.243,0.058824,72.510686,72.906522,0
9,65.329,65.022,0.058824,72.875314,72.346377,0


In [ ]:
y = 

In [87]:
y

Driver
NOR    71.709254
PIA    71.253629
VER    71.685729
HAD          NaN
LEC    72.110986
HAM    71.587143
RUS    71.262814
ANT          NaN
ALO    72.906522
STR    72.346377
ALB    72.364594
SAI    71.312500
OCO    72.191714
BEA          NaN
GAS    72.077929
COL          NaN
HUL    71.978086
BOR          NaN
PER    72.017786
BOT    72.481957
LAW          NaN
LIN          NaN
Name: LapTime (s), dtype: float64